# Chapter 4 Exercise — Coffee Maker Functional Decomposition

Add a functional layer and a completeness judgment to your coffee maker model.

## Problem

The `BrewUnit` applies hot water to grounds. Model that as a new action
nested inside `Brew` (Chapter 1's own action, reopened here — not
recreated) and write an honestly scoped completeness judgment about it.

Work through these steps:

1. Add a new nested action def named `ApplyWater` — genuinely new, not a
   second `action def Brew` (Chapter 1 already declares `action def Brew {
   in beans : Beans; out coffee : Coffee; }` at package level; redeclaring
   it collides). `ApplyWater` takes `in beans : Beans`, `in water :
   ISQ::MassValue[0..*]` (the water supplied), and `in duration :
   ISQ::DurationValue[0..*]`, and produces `out coffee : Coffee`, `out
   delivered : ISQ::MassValue`, and `out retained : ISQ::MassValue`, with
   an asserted balance constraint: `delivered >= 0.0 [SI::kg] and retained
   >= 0.0 [SI::kg] and delivered + retained <= water` (`delivered` is water
   that becomes the coffee liquid output; `retained` is water absorbed by
   the spent grounds — a genuine mass balance, no efficiency or conversion
   formula assumed, mirroring `ApplyHeat`'s own "no specific efficiency is
   assumed"). Nest it inside `Brew` via a reopened body — `first start;
   then action applyWater : ApplyWater { in beans = Brew::beans; } then
   done;` — mirroring `ToastBread`'s reopened body exactly. Do not add a
   `HeatRate`-style expression; there is none in main Chapter 4 either.
2. Add three signal item defs — `BrewStart`, `BrewFinish`, `BrewCancel` —
   each with a `doc` stating what it denotes and explicitly what it is NOT
   (a signal, not the beans or coffee material flow), mirroring `Start`,
   `Finish`, and `Cancel` exactly. `Beans` and `Coffee` already exist from
   Chapter 1 and don't need re-declaring; do not add material item defs for
   grounds or hot water, and do not declare `item def Brew` — it would
   collide with the action def of the same name.
3. Write an `asserted_inference` ReviewRecord (`AI-C04-EX`) claiming that
   THIS ONE nested action's own flows are accounted for and its balance
   constraint is real and evaluable — honestly scoped the same way `AI-C04`
   is scoped for `ApplyHeat` (read `03-completeness-check.ipynb`'s full
   record: its own `claim` states the question is "narrower than whether
   the toaster's functional architecture is complete," and its own
   `counterevidence` states plainly "ApplyHeat is the only function
   modeled"). Do **not** claim that brewing as a whole is functionally
   decomposed. Set `premises` to `["AS-C03-EX"]`.

Verify with `model.ok == True` and `validate_record(inference_record) == []`.


In [ ]:
import opensysml
from toaster.evidence import ReviewRecord, hash_content, validate_record

conn = opensysml.connect(version="v0.9.0")

# Paste your completed Ch3 model here, then add the ApplyWater action def and
# nest it inside Brew's own reopened body (edit Brew's existing declaration
# to add the first/then sequence -- do not redeclare `action def Brew`).
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
print(f"Model ok: {model.ok}")

action = model.find("CoffeeDemo::ApplyWater")
if action:
    print(f"ApplyWater kind: {action.kind}")


In [ ]:
# Step 2: add the three signal item defs.
source_with_items = """
# Your solution here (extend source above)
"""

model2 = conn.load_from_content(source_with_items, strict=False)
print(f"Model with items ok: {model2.ok}")
for name in ("CoffeeDemo::BrewStart", "CoffeeDemo::BrewFinish", "CoffeeDemo::BrewCancel"):
    sym = model2.find(name)
    if sym:
        print(f"{sym.id}: kind={sym.kind}")


In [ ]:
# Step 3: write the asserted_inference record, honestly scoped to this one
# nested action -- not to whether the coffee maker as a whole is
# functionally decomposed (mirror AI-C04's own scoping for ApplyHeat).
inference_record = ReviewRecord(
    identifier="AI-C04-EX",
    kind="asserted_inference",
    claim="# ApplyWater's typed flows account for beans, water and duration in, and coffee, delivered and retained out; the asserted balance constraint (delivered and retained each non-negative, their sum bounded by water) is a real, evaluable relation, not merely declared syntax. Scope this to ApplyWater alone -- not to whether brewing as a whole is decomposed.",
    model_ref="CoffeeDemo::ApplyWater",
    content_hash=hash_content(source_with_items),
    scope="CoffeeDemo::ApplyWater, nested inside CoffeeDemo::Brew",
    criteria="# What would make THIS ONE action's flow accounting complete -- not the criterion for the coffee maker's full functional architecture (mirror AI-C04's own scope note distinguishing ApplyHeat from the toaster's whole functional architecture)",
    premises=["AS-C03-EX"],
    assumption_refs=["# State plainly whether duration plays any role in the balance constraint -- if not, say so, mirroring AI-C04's own note that duration is declared but unconnected to any computation"],
    evidence_refs=["# The balance constraint's own probe results (a plausible split holds; an overdrawn or negative-retained split fails), and confirmation that ApplyWater resolves as a step of Brew via model.find()"],
    rationale="# Why the flow accounting is real for ApplyWater specifically, not for brewing as a whole",
    counterevidence="# What ApplyWater does NOT show: water is unbound (no water source modeled yet), duration is unconnected to any computation, and ApplyWater is the only function modeled -- this is not a complete functional decomposition of the coffee maker",
    residual_uncertainties="# What is not modeled: BrewStart/BrewFinish/BrewCancel are declared but not wired as accepted or produced items of ApplyWater; extraction rate and efficiency belong to a logical component this chapter does not build",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(inference_record)
print(f"Validation errors: {errors}")
conn.close()
